In [0]:
%sql
-- Create medallion schemas and staging volume
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.quarantine;
CREATE SCHEMA IF NOT EXISTS workspace.gold;
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_landing;

-- Enable timestamp_ntz support for 2024+ TLC parquet files
CREATE TABLE IF NOT EXISTS workspace.bronze.yellow_trips_raw
USING DELTA
TBLPROPERTIES (
  'delta.feature.timestampNtz' = 'supported',
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7'
);

In [0]:
%python
# Load static TLC zone lookup CSV into Silver
zone_path = "/Volumes/workspace/bronze/raw_landing/taxi_zone_lookup.csv"

zones_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(zone_path)
)

zones_df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.taxi_zone_lookup")
display(spark.table("workspace.silver.taxi_zone_lookup").limit(5))

In [0]:
%python
dbutils.widgets.text("year_month", "2024-12", "Year-Month (YYYY-MM)")
dbutils.widgets.text("run_id", "manual-run-01", "Pipeline Run ID")

ym = dbutils.widgets.get("year_month")
run_id = dbutils.widgets.get("run_id")
src_file = f"/Volumes/workspace/bronze/raw_landing/yellow_tripdata_{ym}.parquet"

# Idempotent file load: COPY INTO skips files already loaded into the target table
spark.sql(f"""
COPY INTO workspace.bronze.yellow_trips_raw
FROM (
  SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at,
    '{run_id}' AS _pipeline_run_id,
    '{ym}' AS _year_month
  FROM '{src_file}'
)
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true')
""")

Bronze layer complete for 2024-12! Rows in Bronze for 2024-12: 3,668,371


In [0]:
%python
from pyspark.sql import functions as F

ym = dbutils.widgets.get("year_month")

# Cast types, fill nullable fee columns, and generate deterministic trip_id hash
bronze_df = (
    spark.table("workspace.bronze.yellow_trips_raw").filter(F.col("_year_month") == ym)
    .selectExpr(
        "CAST(VendorID AS INT) AS vendor_id", "CAST(tpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime",
        "CAST(tpep_dropoff_datetime AS TIMESTAMP) AS dropoff_datetime", "COALESCE(CAST(passenger_count AS INT), 1) AS passenger_count",
        "CAST(trip_distance AS DOUBLE) AS trip_distance", "CAST(RatecodeID AS INT) AS rate_code_id",
        "COALESCE(CAST(store_and_fwd_flag AS STRING), 'N') AS store_and_fwd_flag",
        "CAST(PULocationID AS INT) AS pu_location_id", "CAST(DOLocationID AS INT) AS do_location_id",
        "CAST(payment_type AS INT) AS payment_type", "CAST(fare_amount AS DOUBLE) AS fare_amount",
        "COALESCE(CAST(extra AS DOUBLE), 0.0) AS extra", "COALESCE(CAST(mta_tax AS DOUBLE), 0.0) AS mta_tax",
        "COALESCE(CAST(tip_amount AS DOUBLE), 0.0) AS tip_amount", "COALESCE(CAST(tolls_amount AS DOUBLE), 0.0) AS tolls_amount",
        "COALESCE(CAST(improvement_surcharge AS DOUBLE), 0.0) AS improvement_surcharge", "CAST(total_amount AS DOUBLE) AS total_amount",
        "COALESCE(CAST(congestion_surcharge AS DOUBLE), 0.0) AS congestion_surcharge", "COALESCE(CAST(Airport_fee AS DOUBLE), 0.0) AS airport_fee",
        "_source_file", "_ingested_at", "_pipeline_run_id", "_year_month"
    )
    .withColumn("trip_duration_minutes", F.round((F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60.0, 2))
    .withColumn("trip_id", F.xxhash64("vendor_id", "pickup_datetime", "dropoff_datetime", "pu_location_id", "do_location_id", "fare_amount", "total_amount", "trip_distance").cast("string"))
)

# Tag rows against DQ rules; empty array means valid record
dq_df = bronze_df.withColumn("failed_rules", F.array_compact(F.array(
    F.when(F.expr("pickup_datetime IS NULL OR dropoff_datetime IS NULL OR pu_location_id IS NULL OR do_location_id IS NULL"), F.lit("RULE_1_NULL_KEYS")),
    F.when(F.expr("dropoff_datetime <= pickup_datetime OR trip_duration_minutes > 1440"), F.lit("RULE_2_INVALID_TIMESTAMPS")),
    F.when(F.expr("trip_distance <= 0 OR trip_distance > 500"), F.lit("RULE_3_INVALID_DISTANCE")),
    F.when(F.expr("fare_amount <= 0 OR total_amount <= 0"), F.lit("RULE_4_INVALID_FARE")),
    F.when(F.date_format("pickup_datetime", "yyyy-MM") != F.lit(ym), F.lit("RULE_5_OUT_OF_PERIOD"))
)))

# Partition overwrite via replaceWhere guarantees idempotent re-runs per month
dq_df.filter("size(failed_rules) > 0").withColumn("_quarantined_at", F.current_timestamp()).write.format("delta").mode("overwrite").option("replaceWhere", f"_year_month = '{ym}'").saveAsTable("workspace.quarantine.yellow_trips_quarantine")

zones = spark.table("workspace.silver.taxi_zone_lookup")
pu = F.broadcast(zones.selectExpr("LocationID AS pu_location_id", "Borough AS pu_borough", "Zone AS pu_zone"))
do = F.broadcast(zones.selectExpr("LocationID AS do_location_id", "Borough AS do_borough", "Zone AS do_zone"))

dq_df.filter("size(failed_rules) = 0").drop("failed_rules").join(pu, "pu_location_id", "left").join(do, "do_location_id", "left").withColumn("_processed_at", F.current_timestamp()).write.format("delta").mode("overwrite").option("replaceWhere", f"_year_month = '{ym}'").saveAsTable("workspace.silver.yellow_trips_clean")

Done 2024-12!


In [0]:
%python
from pyspark.sql import functions as F

# Prepare base dimensions from Silver for Gold reporting tables
silver_df = (
    spark.table("workspace.silver.yellow_trips_clean")
    .withColumn("pickup_borough", F.coalesce("pu_borough", F.lit("Unknown")))
    .withColumn("pickup_zone", F.coalesce("pu_zone", F.lit("Unknown")))
    .withColumn("dropoff_borough", F.coalesce("do_borough", F.lit("Unknown")))
    .withColumn("dropoff_zone", F.coalesce("do_zone", F.lit("Unknown")))
    .withColumn("year_month", F.col("_year_month"))
    .withColumn("trip_date", F.to_date("pickup_datetime"))
    .withColumn("day_of_week_num", F.dayofweek("pickup_datetime"))
    .withColumn("day_of_week_name", F.date_format("pickup_datetime", "EEEE"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
)

agg_exprs = [
    F.count("trip_id").alias("total_trips"),
    F.round(F.sum("passenger_count"), 0).alias("total_passengers"),
    F.round(F.sum("trip_distance"), 2).alias("total_distance_miles"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    F.round(F.sum("fare_amount"), 2).alias("total_fare_amount"),
    F.round(F.sum("tip_amount"), 2).alias("total_tip_amount"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    F.round(F.avg("total_amount"), 2).alias("avg_revenue_per_trip"),
    F.current_timestamp().alias("_updated_at")
]

silver_df.groupBy("trip_date", "year_month", "pickup_borough").agg(*agg_exprs).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.daily_revenue_by_borough")
silver_df.groupBy("year_month", "day_of_week_num", "day_of_week_name", "pickup_hour", "pickup_borough").agg(*agg_exprs).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.hourly_demand_patterns")
silver_df.groupBy("year_month", "pu_location_id", "pickup_borough", "pickup_zone", "do_location_id", "dropoff_borough", "dropoff_zone").agg(*agg_exprs).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.top_od_zone_pairs")

All 3 Gold tables built!


In [0]:
%sql
-- Verify row conservation: bronze_rows = silver_valid_rows +quarantined_rows
WITH bronze_counts AS (
  SELECT _year_month, COUNT(*) AS bronze_rows
  FROM workspace.bronze.yellow_trips_raw
  GROUP BY _year_month
),
silver_counts AS (
  SELECT _year_month, COUNT(*) AS silver_valid_rows
  FROM workspace.silver.yellow_trips_clean
  GROUP BY _year_month
),
quarantine_counts AS (
  SELECT _year_month, COUNT(*) AS quarantined_rows
  FROM workspace.quarantine.yellow_trips_quarantine
  GROUP BY _year_month
)
SELECT
  b._year_month AS year_month,
  b.bronze_rows,
  s.silver_valid_rows,
  q.quarantined_rows,
  (s.silver_valid_rows + q.quarantined_rows) AS total_accounted,
  ROUND((q.quarantined_rows * 100.0) / b.bronze_rows, 2) AS quarantine_rate_pct
FROM bronze_counts b
LEFT JOIN silver_counts s ON b._year_month = s._year_month
LEFT JOIN quarantine_counts q ON b._year_month = q._year_month
ORDER BY year_month;

year_month,bronze_rows,silver_valid_rows,quarantined_rows,total_accounted,quarantine_rate_pct
2024-10,3833771,3681570,152199,3833769,3.97
2024-11,3646369,3509693,136676,3646369,3.75
2024-12,3668371,3518495,149876,3668371,4.09
2025-01,3475226,3252359,222867,3475226,6.41


In [0]:
%sql
-- Breakdown of records rejected by each data quality rule
SELECT
  rule_failed,
  COUNT(*) AS rejected_records
FROM workspace.quarantine.yellow_trips_quarantine
LATERAL VIEW explode(failed_rules) AS rule_failed
GROUP BY rule_failed
ORDER BY rejected_records DESC;

rule_failed,rejected_records
RULE_4_INVALID_FARE,375910
RULE_3_INVALID_DISTANCE,316680
RULE_2_INVALID_TIMESTAMPS,6204
RULE_5_OUT_OF_PERIOD,146


In [0]:
%sql
-- Sample daily revenu metrics by borough
SELECT
  trip_date,
  year_month,
  pickup_borough,
  total_trips,
  total_passengers,
  total_distance_miles,
  avg_distance_miles,
  avg_duration_minutes,
  total_revenue,
  avg_revenue_per_trip
FROM workspace.gold.daily_revenue_by_borough
WHERE pickup_borough != 'Unknown'
ORDER BY trip_date DESC, total_revenue DESC
LIMIT 15;

trip_date,year_month,pickup_borough,total_trips,total_passengers,total_distance_miles,avg_distance_miles,avg_duration_minutes,total_revenue,avg_revenue_per_trip
2025-01-31,2025-01,Manhattan,109071,133869,247894.15,2.27,13.72,2516271.63,23.07
2025-01-31,2025-01,Queens,8510,10834,99486.77,11.69,36.18,587500.57,69.04
2025-01-31,2025-01,Brooklyn,2505,2583,13197.93,5.27,28.56,75293.79,30.06
2025-01-31,2025-01,Bronx,635,640,4744.12,7.47,33.99,21719.71,34.2
2025-01-31,2025-01,N/A,15,22,159.35,10.62,16.93,1303.03,86.87
2025-01-31,2025-01,Staten Island,5,5,32.47,6.49,14.43,138.16,27.63
2025-01-30,2025-01,Manhattan,111993,132744,254614.59,2.27,14.03,2655337.63,23.71
2025-01-30,2025-01,Queens,8393,10681,100101.92,11.93,37.0,589179.53,70.2
2025-01-30,2025-01,Brooklyn,2088,2151,11317.49,5.42,30.57,63248.64,30.29
2025-01-30,2025-01,Bronx,484,494,3789.46,7.83,36.59,17028.41,35.18
